# OAT 3 — Random Forest e Gradient Boosting

Nesta etapa, a equipe irá testar os modelos **Random Forest Regressor** e **Gradient Boosting Regressor** para previsão diária de `Trocas_Oleo`.

Os modelos serão comparados com o **baseline Naive** desenvolvido anteriormente. A divisão dos dados será cronológica, sem embaralhamento, para evitar que informações do futuro sejam utilizadas no treinamento.

### Decisões da equipe

1. **Algoritmos:** Random Forest Regressor e Gradient Boosting Regressor.
2. **Configuração inicial:** hiperparâmetros padrão do scikit-learn.
3. **Features:** lags, médias móveis, dia da semana e mês.
4. **Divisão temporal:** 80% mais antigos para treino e 20% mais recentes para teste.
5. **Data leakage:** todas as etapas que aprendem parâmetros serão ajustadas somente com `X_train` e `y_train`.
6. **Métricas:** RMSE e MAPE.
7. **Comparação:** os dois modelos serão comparados entre si e com o baseline Naive.


## 1. Importação das bibliotecas

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.pipeline import Pipeline


## 2. Carregamento e organização temporal

A série principal continua sendo `Trocas_Oleo`. Os dados são ordenados por data antes da criação das variáveis históricas.

In [ ]:
data_path = next(
    (
        path for path in [
            Path("data/mecaniqa_dataset.xlsx"),
            Path("../data/mecaniqa_dataset.xlsx"),
        ]
        if path.exists()
    ),
    None,
)

if data_path is None:
    raise FileNotFoundError(
        "Dataset não encontrado. Abra o projeto pela pasta mecaniQA-salvador."
    )

dados_brutos = pd.read_excel(data_path)

dados_brutos["Data"] = pd.to_datetime(
    dados_brutos["Data"],
    errors="raise"
)

serie = (
    dados_brutos
    .sort_values("Data")
    .set_index("Data")["Trocas_Oleo"]
    .resample("D")
    .sum(min_count=1)
)

serie.head()


## 3. Feature engineering temporal

As variáveis históricas utilizam `shift(1)` quando a informação precisa representar somente o passado. Dessa forma, a previsão de um determinado dia não utiliza o valor observado daquele próprio dia.

As features utilizadas são:

- `lag_1`: valor do dia anterior;
- `lag_7`: valor de sete dias antes;
- `lag_30`: valor de trinta dias antes;
- `rolling_7`: média dos sete dias anteriores;
- `rolling_30`: média dos trinta dias anteriores;
- `dia_semana`: dia da semana;
- `mes`: mês.


In [ ]:
df = pd.DataFrame({"alvo": serie})

df["lag_1"] = df["alvo"].shift(1)
df["lag_7"] = df["alvo"].shift(7)
df["lag_30"] = df["alvo"].shift(30)

df["rolling_7"] = (
    df["alvo"]
    .shift(1)
    .rolling(window=7, min_periods=7)
    .mean()
)

df["rolling_30"] = (
    df["alvo"]
    .shift(1)
    .rolling(window=30, min_periods=30)
    .mean()
)

df["dia_semana"] = df.index.dayofweek
df["mes"] = df.index.month

features = [
    "lag_1",
    "lag_7",
    "lag_30",
    "rolling_7",
    "rolling_30",
    "dia_semana",
    "mes",
]

df = df.dropna(subset=["alvo", *features])

X = df[features]
y = df["alvo"]

print(f"Total de registros utilizados: {len(df)}")
df.head()


## 4. Separação cronológica entre treino e teste

Os dados não serão embaralhados.

- Os **80% registros mais antigos** serão utilizados no treinamento.
- Os **20% registros mais recentes** serão utilizados exclusivamente no teste.

Assim, a avaliação representa uma situação mais próxima de uma previsão real.


In [ ]:
ponto_corte = int(len(X) * 0.80)

X_train = X.iloc[:ponto_corte]
X_test = X.iloc[ponto_corte:]

y_train = y.iloc[:ponto_corte]
y_test = y.iloc[ponto_corte:]

assert X_train.index.max() < X_test.index.min()

print(
    f"Treino: {X_train.index.min().date()} "
    f"a {X_train.index.max().date()} "
    f"({len(X_train)} dias)"
)

print(
    f"Teste:  {X_test.index.min().date()} "
    f"a {X_test.index.max().date()} "
    f"({len(X_test)} dias)"
)


## 5. Modelos escolhidos

A equipe decidiu testar dois algoritmos de ensemble:

### Random Forest Regressor

O Random Forest combina várias árvores de decisão. Isso permite capturar relações não lineares entre as variáveis históricas e a quantidade de trocas de óleo.

### Gradient Boosting Regressor

O Gradient Boosting constrói os modelos sequencialmente, tentando corrigir os erros cometidos pelas árvores anteriores.

Nesta primeira etapa serão utilizados os **hiperparâmetros padrão do scikit-learn**, conforme a decisão da equipe.


In [ ]:
modelo_rf = Pipeline(
    steps=[
        ("imputacao", SimpleImputer(strategy="median")),
        (
            "modelo",
            RandomForestRegressor(
                random_state=42
            ),
        ),
    ]
)

modelo_gb = Pipeline(
    steps=[
        ("imputacao", SimpleImputer(strategy="median")),
        (
            "modelo",
            GradientBoostingRegressor(
                random_state=42
            ),
        ),
    ]
)


## 6. Treinamento

O `fit()` é executado somente sobre os dados de treinamento.

Como o `SimpleImputer` está dentro do `Pipeline`, qualquer estatística necessária para a imputação é aprendida somente a partir de `X_train`. O conjunto de teste não participa do treinamento.


In [ ]:
modelo_rf.fit(X_train, y_train)
modelo_gb.fit(X_train, y_train)

print("Random Forest treinado.")
print("Gradient Boosting treinado.")


## 7. Previsões no período de teste

In [ ]:
previsao_rf = modelo_rf.predict(X_test)
previsao_gb = modelo_gb.predict(X_test)

resultado = pd.DataFrame(
    {
        "Real": y_test,
        "Random Forest": previsao_rf,
        "Gradient Boosting": previsao_gb,
    },
    index=y_test.index,
)

resultado.head()


## 8. Avaliação — RMSE e MAPE

O **RMSE** mede o tamanho dos erros de previsão, dando maior peso aos erros maiores.

O **MAPE** representa o erro percentual médio. Como o MAPE não deve ser calculado diretamente sobre valores reais iguais a zero, esses registros são retirados somente do cálculo dessa métrica.


In [ ]:
def calcular_metricas(real, previsto):
    pares = pd.DataFrame(
        {
            "real": real,
            "previsto": previsto,
        }
    ).dropna()

    rmse = np.sqrt(
        mean_squared_error(
            pares["real"],
            pares["previsto"]
        )
    )

    sem_zero = pares["real"].ne(0)

    if sem_zero.any():
        mape = (
            100
            * mean_absolute_percentage_error(
                pares.loc[sem_zero, "real"],
                pares.loc[sem_zero, "previsto"],
            )
        )
    else:
        mape = np.nan

    return rmse, mape


rmse_rf, mape_rf = calcular_metricas(y_test, previsao_rf)
rmse_gb, mape_gb = calcular_metricas(y_test, previsao_gb)

metricas = pd.DataFrame(
    {
        "Modelo": [
            "Random Forest",
            "Gradient Boosting",
        ],
        "RMSE": [
            rmse_rf,
            rmse_gb,
        ],
        "MAPE (%)": [
            mape_rf,
            mape_gb,
        ],
    }
).set_index("Modelo")

metricas


## 9. Baseline Naive

O Naive prevê o valor do dia atual utilizando o último valor conhecido.

Ele é mantido como referência para verificar se os modelos de Machine Learning realmente melhoram a previsão.


In [ ]:
baseline_naive = serie.ffill().shift(1)

baseline_teste = baseline_naive.reindex(y_test.index)

rmse_naive, mape_naive = calcular_metricas(
    y_test,
    baseline_teste
)

metricas_comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Naive",
            "Random Forest",
            "Gradient Boosting",
        ],
        "RMSE": [
            rmse_naive,
            rmse_rf,
            rmse_gb,
        ],
        "MAPE (%)": [
            mape_naive,
            mape_rf,
            mape_gb,
        ],
    }
).set_index("Modelo")

metricas_comparacao


## 10. Comparação dos resultados

Quanto menor o **RMSE** e o **MAPE**, melhor o desempenho do modelo.

A comparação abaixo permite verificar se Random Forest e Gradient Boosting conseguem reduzir os erros em relação ao baseline Naive.


In [ ]:
melhor_rmse = metricas_comparacao["RMSE"].idxmin()
melhor_mape = metricas_comparacao["MAPE (%)"].idxmin()

print("Resultado da comparação")
print("=" * 40)
print(f"Melhor RMSE: {melhor_rmse}")
print(f"Melhor MAPE: {melhor_mape}")

metricas_comparacao.sort_values("RMSE")


## 11. Visualização — valores reais x previsões

In [ ]:
ax = resultado.plot(
    figsize=(14, 6),
    linewidth=2,
)

ax.set_title(
    "Trocas de óleo — Real x Random Forest x Gradient Boosting"
)
ax.set_xlabel("Data")
ax.set_ylabel("Quantidade de trocas de óleo")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


## 12. Verificação contra Data Leakage

O experimento respeita a linha do tempo:

- os dados foram ordenados por data;
- não houve embaralhamento;
- as variáveis de histórico usam somente informações anteriores;
- o corte entre treino e teste foi feito cronologicamente;
- `fit()` foi aplicado somente ao conjunto de treino;
- o conjunto de teste foi utilizado apenas para gerar previsões e calcular as métricas;
- a imputação está dentro do pipeline e é ajustada somente com `X_train`.

Dessa forma, os modelos são avaliados utilizando uma situação próxima daquela que ocorreria quando novas datas fossem realmente previstas.


## 13. Conclusão

Nesta etapa foram treinados os modelos **Random Forest Regressor** e **Gradient Boosting Regressor** utilizando inicialmente os hiperparâmetros padrão do scikit-learn.

Os modelos utilizaram informações temporais e históricas das trocas de óleo. Os resultados foram avaliados por **RMSE** e **MAPE** e comparados com o baseline **Naive**.

A conclusão final deve ser preenchida após a execução do notebook, considerando qual modelo apresentou os menores erros e se houve melhoria em relação ao baseline.
